# H6b: Activation-dependent Muon vs SGD Final-Loss Comparison Under Separately Tuned Learning Rates

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H6b_RESIDUAL_DIRECTIONAL_ADVANTAGE/run_experiment.py`

## Scope of this notebook
This notebook is the presentation and analysis layer for the counterpart script. It **imports and runs the script's exact experiment implementation** instead of re-implementing the training loop, so the notebook and script stay aligned.

### What is measured here
- For each activation (`linear`, `relu`, `tanh`, `gelu`), separately tune learning rate for Muon and SGD on **3 sweep seeds**.
- Re-evaluate the selected learning rate on **10 seeds**.
- Compare **mean final loss** and report the activation-level ratio
  `advantage = mean_final_loss_SGD / mean_final_loss_Muon`.

### What is *not* directly measured here
- update-direction alignment
- singular-value rescue or spectral conditioning diagnostics
- a causal mechanism for why any loss gap appears

This is therefore best read as an **activation-dependent final-loss comparison in a small synthetic regression toy problem**, not as a direct measurement of “directional advantage.” The linear case is only an **internal control inside this exact protocol**; whether it reproduces any earlier `~7x` baseline must be checked empirically below, not assumed in advance.

## 1. Notebook-safe imports and counterpart loading

The original notebook duplicated the training code and relied on `__file__`, which is fragile in Jupyter. This version resolves the project root from the current working directory, loads the counterpart script explicitly, and uses its exported `run_experiment()` function.

In [ ]:
from pathlib import Path
import importlib.util
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)

    class Markdown(str):
        pass

EXPERIMENT_REL_PATH = Path(
    'experiments/Tier_1_Core_Mechanism_Tests/'
    'H6b_RESIDUAL_DIRECTIONAL_ADVANTAGE/run_experiment.py'
)


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / EXPERIMENT_REL_PATH).exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate the repository root from the current working '
        f'directory {start}. Expected to find {EXPERIMENT_REL_PATH} in one of '
        'the current directory or its parents.'
    )


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPT_PATH = REPO_ROOT / EXPERIMENT_REL_PATH

spec = importlib.util.spec_from_file_location('h6b_run_experiment_module', SCRIPT_PATH)
h6b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h6b)

identity_df = pd.DataFrame(
    [
        ('repo_root', str(REPO_ROOT)),
        ('script_path', str(SCRIPT_PATH)),
        ('experiment_id', h6b.EXPERIMENT_ID),
        ('script_title', h6b.EXPERIMENT_TITLE),
    ],
    columns=['field', 'value'],
)

display(identity_df)

## 2. Execute the exact counterpart experiment

This cell calls the script's `run_experiment(verbose=True)` function. On this repository and hardware class, the default run is expected to take **a few minutes on CPU**. The returned object contains the raw structured results used throughout the rest of the notebook.

In [ ]:
notebook_wall_start = time.time()
results = h6b.run_experiment(verbose=True)
notebook_wall_seconds = time.time() - notebook_wall_start

config = results['config']
metadata = results['metadata']

print(f'Notebook wall-clock for run_experiment(): {notebook_wall_seconds:.2f}s')
print(f"Script-reported runtime_seconds: {results['runtime_seconds']:.2f}s")

## 3. Reproducibility, runtime, and configuration

The experiment is intentionally small and fully deterministic given the hard-coded seeds. The table below records the exact configuration, learning-rate grids, and seed protocol used by the imported script.

In [ ]:
repro_rows = [
    ('title', metadata['title']),
    ('scope', metadata['scope']),
    ('selection_rule', metadata['selection_rule']),
    ('runtime_seconds_reported_by_script', round(results['runtime_seconds'], 2)),
    ('runtime_seconds_measured_in_notebook', round(notebook_wall_seconds, 2)),
    ('activations', ', '.join(config['activations'])),
    ('optimizers', ', '.join(config['optimizers'])),
    ('network', f"{config['num_layers']}-layer square net, dim={config['dim']}"),
    ('num_steps', config['num_steps']),
    ('batch_size', config['batch_size']),
    ('num_seeds', config['num_seeds']),
    ('sweep_num_seeds', config['sweep_num_seeds']),
    ('momentum', config['momentum']),
    ('ns_iters', config['ns_iters']),
    ('divergence_threshold', config['divergence_threshold']),
    ('muon_lrs', ', '.join(map(str, config['muon_lrs']))),
    ('sgd_lrs', ', '.join(map(str, config['sgd_lrs']))),
    ('all_seeds', ', '.join(map(str, results['seeds']))),
    ('sweep_seeds', ', '.join(map(str, results['sweep_seeds']))),
]

repro_df = pd.DataFrame(repro_rows, columns=['field', 'value'])
display(repro_df)

seed_roles_df = pd.DataFrame({
    'seed': results['seeds'],
    'used_in_lr_sweep': [seed in set(results['sweep_seeds']) for seed in results['seeds']],
})
display(seed_roles_df)

## 4. Build analysis tables from the returned result object

Everything below is derived from the structured output of `run_experiment()`. No separate experiment loop is implemented in the notebook.

In [ ]:
def build_analysis_tables(results_dict):
    sweep_records = []
    selected_records = []
    seed_records = []
    summary_records = []

    for activation, activation_payload in results_dict['results_by_activation'].items():
        optimizer_payloads = activation_payload['optimizers']

        for optimizer, optimizer_payload in optimizer_payloads.items():
            for row in optimizer_payload['lr_sweep']:
                sweep_records.append({
                    'activation': activation,
                    'optimizer': optimizer,
                    'lr': row['lr'],
                    'mean_loss': row['mean_loss'],
                    'std_loss': row['std_loss'],
                    'n_diverged': row['n_diverged'],
                    'n_finite': row['n_finite'],
                    'n_total': row['n_total'],
                    'selected': row['selected'],
                })

            full_eval = optimizer_payload['full_eval']
            selected_records.append({
                'activation': activation,
                'optimizer': optimizer,
                'selected_lr': optimizer_payload['selected_lr'],
                'sweep_mean_loss_at_selected_lr': optimizer_payload['selected_lr_sweep_mean_loss'],
                'full_mean_loss': full_eval['mean_loss'],
                'full_std_loss': full_eval['std_loss'],
                'full_n_diverged': full_eval['n_diverged'],
                'full_n_total': full_eval['n_total'],
            })

            for seed_row in full_eval['seed_records']:
                seed_records.append({
                    'activation': activation,
                    'optimizer': optimizer,
                    'seed': seed_row['seed'],
                    'final_loss': seed_row['final_loss'],
                    'is_finite': np.isfinite(seed_row['final_loss']),
                })

        muon_payload = optimizer_payloads['muon']['full_eval']
        sgd_payload = optimizer_payloads['sgd']['full_eval']
        summary_records.append({
            'activation': activation,
            'muon_lr': optimizer_payloads['muon']['selected_lr'],
            'sgd_lr': optimizer_payloads['sgd']['selected_lr'],
            'muon_mean_loss': muon_payload['mean_loss'],
            'muon_std_loss': muon_payload['std_loss'],
            'muon_n_diverged': muon_payload['n_diverged'],
            'sgd_mean_loss': sgd_payload['mean_loss'],
            'sgd_std_loss': sgd_payload['std_loss'],
            'sgd_n_diverged': sgd_payload['n_diverged'],
            'advantage': activation_payload['advantage'],
        })

    sweep_df = pd.DataFrame(sweep_records)
    selected_df = pd.DataFrame(selected_records)
    seed_df = pd.DataFrame(seed_records)
    summary_df = pd.DataFrame(summary_records)

    test_records = []
    for test_name, test_payload in results_dict['tests'].items():
        observed = ''
        if test_name == 'T3':
            observed = (
                f"tanh={test_payload['observed_tanh_advantage']:.3f}x; "
                f"relu={test_payload['observed_relu_advantage']:.3f}x"
            )
        else:
            observed = ', '.join(
                f"{act}={adv:.3f}x" for act, adv in results_dict['advantages'].items()
            )
        test_records.append({
            'test': test_name,
            'passed': bool(test_payload['passed']),
            'criterion': test_payload['criterion'],
            'description': test_payload['description'],
            'observed': observed,
        })
    tests_df = pd.DataFrame(test_records)
    return sweep_df, selected_df, seed_df, summary_df, tests_df


sweep_df, selected_df, seed_df, summary_df, tests_df = build_analysis_tables(results)

summary_df

## 5. Learning-rate sweep summary

The next table and figure show exactly what the script optimized over on the **3 sweep seeds**. This is an honest view of the proxy actually used for learning-rate selection: the selected LR is the one with the **lowest mean finite final loss**, while divergence counts are recorded but not separately penalized beyond the all-diverged case.

In [ ]:
selected_sweep_df = (
    sweep_df[sweep_df['selected']]
    .sort_values(['activation', 'optimizer'])
    .reset_index(drop=True)
)
selected_sweep_df

In [ ]:
activation_order = config['activations']
optimizer_order = config['optimizers']
colors = {'muon': 'tab:blue', 'sgd': 'tab:orange'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
axes = axes.ravel()

for ax, activation in zip(axes, activation_order):
    activation_subset = sweep_df[sweep_df['activation'] == activation]
    finite_means = activation_subset[np.isfinite(activation_subset['mean_loss'])]['mean_loss']
    y_top = finite_means.max() * 1.4 if len(finite_means) else 1.0

    for optimizer in optimizer_order:
        subset = (
            activation_subset[activation_subset['optimizer'] == optimizer]
            .sort_values('lr')
            .copy()
        )
        y_values = subset['mean_loss'].where(np.isfinite(subset['mean_loss']), np.nan)
        ax.plot(
            subset['lr'],
            y_values,
            marker='o',
            linewidth=2,
            label=optimizer.upper(),
            color=colors[optimizer],
        )

        selected_subset = subset[subset['selected']]
        if not selected_subset.empty:
            ax.scatter(
                selected_subset['lr'],
                selected_subset['mean_loss'].where(np.isfinite(selected_subset['mean_loss']), np.nan),
                marker='*',
                s=220,
                color=colors[optimizer],
                edgecolor='black',
                zorder=5,
            )

        for _, row in subset.iterrows():
            if np.isfinite(row['mean_loss']):
                ax.annotate(
                    f"d={int(row['n_diverged'])}",
                    (row['lr'], row['mean_loss']),
                    textcoords='offset points',
                    xytext=(0, 6),
                    ha='center',
                    fontsize=8,
                    color=colors[optimizer],
                )
            else:
                ax.scatter(row['lr'], y_top, marker='x', s=70, color=colors[optimizer])
                ax.annotate(
                    f"inf,d={int(row['n_diverged'])}",
                    (row['lr'], y_top),
                    textcoords='offset points',
                    xytext=(0, 6),
                    ha='center',
                    fontsize=8,
                    color=colors[optimizer],
                )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'{activation}: LR sweep on {config["sweep_num_seeds"]} seeds')
    ax.set_xlabel('learning rate')
    ax.set_ylabel('mean final loss')
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle(
    'LR sweep curves by activation and optimizer\n'
    'Stars mark the selected LR; point annotations show number of diverged sweep seeds.',
    fontsize=14,
)
plt.show()

### Interpretation of the LR sweep figure

These curves should be read as **hyperparameter-selection diagnostics**, not as a mechanistic analysis. They show whether Muon and SGD prefer different learning-rate regions and whether the selected LR was chosen from a broad stable basin or a sharper stability-limited region. Because the current selection rule minimizes **mean finite** loss, divergence counts should still be inspected directly when interpreting the selected LR.

## 6. Final-loss comparison across the 10 evaluation seeds

The next table summarizes the actual metric used for the activation comparison: full-evaluation mean final loss, seed-to-seed standard deviation, divergence counts, and the resulting SGD/Muon advantage ratio.

In [ ]:
summary_df = summary_df.sort_values('activation').reset_index(drop=True)
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)

x_positions = []
x_labels = []
position = 0
rng = np.random.RandomState(0)

for activation in activation_order:
    for optimizer in optimizer_order:
        subset = seed_df[
            (seed_df['activation'] == activation) &
            (seed_df['optimizer'] == optimizer)
        ]
        x = position
        x_positions.append(x)
        x_labels.append(f'{activation}\n{optimizer.upper()}')

        finite_subset = subset[subset['is_finite']]
        if not finite_subset.empty:
            jitter = rng.uniform(-0.08, 0.08, size=len(finite_subset))
            ax.scatter(
                np.full(len(finite_subset), x) + jitter,
                finite_subset['final_loss'],
                alpha=0.75,
                color=colors[optimizer],
                s=35,
            )
            mean_loss = finite_subset['final_loss'].mean()
            std_loss = finite_subset['final_loss'].std(ddof=0)
            ax.errorbar(
                [x],
                [mean_loss],
                yerr=[[std_loss], [std_loss]],
                fmt='o',
                color='black',
                capsize=5,
                markersize=7,
                linewidth=1.8,
            )

        n_diverged = int((~subset['is_finite']).sum())
        if n_diverged:
            y_level = finite_subset['final_loss'].max() if not finite_subset.empty else 1.0
            ax.annotate(
                f'diverged: {n_diverged}/{len(subset)}',
                (x, y_level),
                textcoords='offset points',
                xytext=(0, 10),
                ha='center',
                fontsize=8,
                color='red',
            )

        position += 1
    position += 0.8

ax.set_yscale('log')
ax.set_xticks(x_positions)
ax.set_xticklabels(x_labels)
ax.set_ylabel('final loss across full evaluation seeds')
ax.set_title('Per-seed final losses with mean ± std overlay')
ax.grid(True, axis='y', alpha=0.3)
plt.show()

### Interpretation of the final-loss comparison

This figure is the most direct view of the experiment's core output. The point cloud shows the **distribution across seeds**, while the black marker and error bar summarize the **mean ± standard deviation** for each activation/optimizer pair. This is still a toy-problem result, but it makes the effect size and variability much more explicit than a single printed ratio.

## 7. Advantage by activation

The bar chart below compresses the full-evaluation comparison into the activation-level ratio actually tested by the script:

`advantage = mean_final_loss_SGD / mean_final_loss_Muon`

Values above `1` indicate lower mean final loss for Muon.

In [ ]:
advantage_df = summary_df[['activation', 'advantage']].copy().sort_values('activation')

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
bars = ax.bar(advantage_df['activation'], advantage_df['advantage'], color='tab:green', alpha=0.8)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1.5, label='parity (1x)')
ax.set_ylabel('SGD mean final loss / Muon mean final loss')
ax.set_title('Activation-level Muon advantage under separately tuned LRs')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

for bar, value in zip(bars, advantage_df['advantage']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f'{value:.2f}x',
        ha='center',
        va='bottom',
        fontsize=10,
    )

plt.show()

## 8. T1 / T2 / T3 verdicts

These verdicts are modest proxies derived from the same final-loss ratios. They are useful as summary checks, but they should not be interpreted as strong mechanistic tests on their own.

In [ ]:
tests_df

In [ ]:
linear_adv = float(results['advantages'].get('linear', np.nan))
relu_adv = float(results['advantages'].get('relu', np.nan))
tanh_adv = float(results['advantages'].get('tanh', np.nan))
gelu_adv = float(results['advantages'].get('gelu', np.nan))

if np.isfinite(linear_adv) and linear_adv >= 5.0:
    linear_control_statement = (
        f'The linear control produced an observed advantage of {linear_adv:.2f}x, '
        'which is at least in the rough order of magnitude of a prior ~7x reference.'
    )
else:
    linear_control_statement = (
        f'The linear control produced an observed advantage of {linear_adv:.2f}x, '
        'so this protocol does not reproduce a ~7x linear-control baseline.'
    )

if np.isfinite(tanh_adv) and np.isfinite(relu_adv) and tanh_adv > relu_adv:
    tanh_vs_relu_statement = (
        f'tanh exceeded ReLU in this run ({tanh_adv:.2f}x vs {relu_adv:.2f}x).'
    )
else:
    tanh_vs_relu_statement = (
        f'tanh did not exceed ReLU in this run ({tanh_adv:.2f}x vs {relu_adv:.2f}x).'
    )

verdict_lines = [
    f"- **T1:** {'PASS' if results['tests']['T1']['passed'] else 'FAIL'}",
    f"- **T2:** {'PASS' if results['tests']['T2']['passed'] else 'FAIL'}",
    f"- **T3:** {'PASS' if results['tests']['T3']['passed'] else 'FAIL'}",
    '',
    f'- {linear_control_statement}',
    f'- {tanh_vs_relu_statement}',
]

display(Markdown('\n'.join(verdict_lines)))

## 9. Data-bound conclusion

The final conclusion below is intentionally tied to the actual returned numbers from the script, rather than to a conditional narrative.

In [ ]:
best_activation = advantage_df.sort_values('advantage', ascending=False).iloc[0]
worst_activation = advantage_df.sort_values('advantage', ascending=True).iloc[0]

linear_phrase = 'does not reproduce' if linear_adv < 5.0 else 'is at least roughly consistent with'
tanh_vs_relu_phrase = (
    'did exceed'
    if (np.isfinite(tanh_adv) and np.isfinite(relu_adv) and tanh_adv > relu_adv)
    else 'did not exceed'
)

conclusion = f"""
### Main empirical takeaway

In this first-pass H6b implementation, Muon is compared against SGD by **mean final loss after separately tuned learning-rate sweeps** across four activations in a small synthetic regression task.

- The strongest observed advantage was for **{best_activation['activation']}** at **{best_activation['advantage']:.2f}x**.
- The weakest observed advantage was for **{worst_activation['activation']}** at **{worst_activation['advantage']:.2f}x**.
- The linear internal control yielded **{linear_adv:.2f}x**, which {linear_phrase} a prior `~7x` reference.
- The tanh-vs-ReLU comparison gave **tanh = {tanh_adv:.2f}x** and **ReLU = {relu_adv:.2f}x**, so tanh {tanh_vs_relu_phrase} ReLU in this run.
- The summary tests were **T1 = {'PASS' if results['tests']['T1']['passed'] else 'FAIL'}**, **T2 = {'PASS' if results['tests']['T2']['passed'] else 'FAIL'}**, and **T3 = {'PASS' if results['tests']['T3']['passed'] else 'FAIL'}**.

### Interpretation limits

These results support only an **activation-dependent final-loss comparison under separately tuned learning rates**. They do **not** directly establish update-direction superiority, singular-value rescue, or a causal mechanism. Any stronger mechanistic claim would require extra diagnostics beyond the current experiment, such as spectral measurements, alignment metrics, or layerwise gradient statistics.
"""

display(Markdown(conclusion))